In [1]:
import torch
import gc

# Очищаем кэш GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


# Принудительный сборщик мусора
gc.collect()

# Проверяем результат
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU memory allocated: 0.00 GB
GPU memory cached: 0.00 GB


In [2]:
import sys
import os
import torch
import numpy as np
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

import torch_pruning as tp
from peft import LoraConfig, get_peft_model

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_SIZE = 2
SEQ_LENGTH = 64
# MODEL_NAME = "google/gemma-2-2b"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model loaded")

Model loaded


# IF LORA

In [4]:
lora_config = LoraConfig(
    r=8,  # rank
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM"
)

# === APPLY LoRA ===
model = get_peft_model(model, lora_config)
for name, param in model.named_parameters():
    if "lora_" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

print("\nLoRA applied")
model.print_trainable_parameters()

RESULTS_DIR = "results/LORA"
os.makedirs(RESULTS_DIR, exist_ok=True)


LoRA applied
trainable params: 3,293,184 || all params: 497,325,952 || trainable%: 0.6622


In [5]:
example_inputs = {
    "input_ids": torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, SEQ_LENGTH)).to(device),
    "attention_mask": torch.ones(BATCH_SIZE, SEQ_LENGTH).to(device),
    "labels": torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, SEQ_LENGTH)).to(device)
}

In [7]:
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

wrapped_model = ModelWrapper(model).to(device)

DG_before = tp.DependencyGraph().build_dependency(
    wrapped_model,
    example_inputs=example_inputs
)

print("DependencyGraph built")

DependencyGraph built


In [8]:
ignored_layers = []

for name, module in model.named_modules():
    if 'embed_tokens' in name or 'lm_head' in name:
        ignored_layers.append(module)

In [9]:
pruning_ratio = 0.5        
pruner = tp.pruner.MetaPruner(
            model=wrapped_model,
            example_inputs=example_inputs,
            importance=tp.importance.MagnitudeImportance(p=2),
            pruning_ratio=pruning_ratio,
            ignored_layers=ignored_layers,
            round_to=8,
            iterative_steps=1,
        )

groups = pruner.DG.get_all_groups()

In [10]:
def compute_group_importance(pruner, group):
    try:
        imp = pruner.importance(group)
        if imp is not None and len(imp) > 0:
            return imp
    except Exception as e:
        print(f"Error computing importance: {e}")
    return None

In [ ]:
all_groups = DG_before.get_all_groups()
layer_mapping = defaultdict(lambda: {"in_indices": set(), "out_indices": set()})
module_to_name = {m: n for n, m in model.named_modules()}

for group_id, group in enumerate(all_groups):
    try:
        importance = pruner.importance(group)
        
        if importance is None:
            continue
            
        if isinstance(importance, torch.Tensor):
            if importance.dim() == 2:
                channel_importance = importance.abs().sum(dim=1)
                n = channel_importance.shape[0]
                k = max(1, int(n * (1 - pruning_ratio)))
                keep_indices = torch.topk(channel_importance, k=k, largest=True).indices
                
                # Сохраняем индексы для всех модулей в группе
                for dep, idxs in group:
                    module = dep.target.module
                    if module in module_to_name:
                        layer_name = module_to_name[module]
                        handler_name = dep.handler.__name__.lower()
                        
                        if 'output' in handler_name or 'out' in handler_name:
                            for idx in keep_indices.tolist():
                                layer_mapping[layer_name]['out_indices'].add(idx)
                        elif 'input' in handler_name or 'in' in handler_name:
                            for idx in keep_indices.tolist():
                                layer_mapping[layer_name]['in_indices'].add(idx)
                                
            elif importance.dim() == 1:
                n = importance.shape[0]
                k = max(1, int(n * (1 - pruning_ratio)))
                keep_indices = torch.topk(importance.abs(), k=k, largest=True).indices
                
                for dep, idxs in group:
                    module = dep.target.module
                    if module in module_to_name:
                        layer_name = module_to_name[module]
                        handler_name = dep.handler.__name__.lower()
                        
                        if 'output' in handler_name or 'out' in handler_name:
                            for idx in keep_indices.tolist():
                                layer_mapping[layer_name]['out_indices'].add(idx)
                        elif 'input' in handler_name or 'in' in handler_name:
                            for idx in keep_indices.tolist():
                                layer_mapping[layer_name]['in_indices'].add(idx)
                                
    except Exception as e:
        print(f"Error processing group {group_id}: {e}")
        continue

In [12]:
pruner.step()

In [13]:
final_mapping = {}
for layer_name, indices in layer_mapping.items():
    final_mapping[layer_name] = {
        "in_indices": sorted(list(indices['in_indices'])),
        "out_indices": sorted(list(indices['out_indices']))
    }

rows = []
for layer_name, indices in final_mapping.items():
    for idx in indices['out_indices']:
        rows.append({"layer_name": layer_name, "type": "output", "index": idx})
    for idx in indices['in_indices']:
        rows.append({"layer_name": layer_name, "type": "input", "index": idx})

df = pd.DataFrame(rows)
df.to_csv("results/correct_kept_indices.csv", index=False)

# Выводим статистику
print("\n" + "="*80)
print("CORRECTED MAPPING STATISTICS:")
print("="*80)
for layer_name, indices in final_mapping.items():
    if 'q_proj' in layer_name and indices['out_indices']:
        print(f"\n{layer_name}:")
        print(f"  Kept output indices: {len(indices['out_indices'])} / 896")
        print(f"  First 10: {indices['out_indices'][:10]}")
        break



CORRECTED MAPPING STATISTICS:

base_model.model.model.layers.23.self_attn.q_proj.base_layer:
  Kept output indices: 448 / 896
  First 10: [10, 16, 18, 21, 23, 24, 25, 26, 27, 29]


In [15]:
df = pd.read_csv('results/correct_kept_indices.csv')
attention_rows = df[df['layer_name'].str.contains('mlp', case=False, na=False)]
attention_rows.head(20)

,layer_name,type,index
76416,base_model.model.model.layers.23.mlp.down_proj...,output,4
76417,base_model.model.model.layers.23.mlp.down_proj...,output,5
76418,base_model.model.model.layers.23.mlp.down_proj...,output,6
76419,base_model.model.model.layers.23.mlp.down_proj...,output,7
76420,base_model.model.model.layers.23.mlp.down_proj...,output,9
76421,base_model.model.model.layers.23.mlp.down_proj...,output,11
76422,base_model.model.model.layers.23.mlp.down_proj...,output,13
76423,base_model.model.model.layers.23.mlp.down_proj...,output,14
76424,base_model.model.model.layers.23.mlp.down_proj...,output,15
76425,base_model.model.model.layers.23.mlp.down_proj...,output,16


In [19]:
unique_layers = df['layer_name'].unique()
print(unique_layers)

['lm_head' 'model.layers.23.mlp.down_proj'
 'model.layers.23.self_attn.o_proj' 'model.layers.23.mlp.gate_proj'
 'model.layers.23.mlp.up_proj' 'model.layers.22.mlp.down_proj'
 'model.layers.23.self_attn.q_proj' 'model.layers.23.self_attn.k_proj'
 'model.layers.23.self_attn.v_proj' 'model.layers.22.self_attn.o_proj'
 'model.layers.22.mlp.gate_proj' 'model.layers.22.mlp.up_proj'
 'model.layers.21.mlp.down_proj' 'model.layers.22.self_attn.q_proj'
 'model.layers.22.self_attn.k_proj' 'model.layers.22.self_attn.v_proj'
 'model.layers.21.self_attn.o_proj' 'model.layers.21.mlp.gate_proj'
 'model.layers.21.mlp.up_proj' 'model.layers.20.mlp.down_proj'
 'model.layers.21.self_attn.q_proj' 'model.layers.21.self_attn.k_proj'
 'model.layers.21.self_attn.v_proj' 'model.layers.20.self_attn.o_proj'
 'model.layers.20.mlp.gate_proj' 'model.layers.20.mlp.up_proj'
 'model.layers.19.mlp.down_proj' 'model.layers.20.self_attn.q_proj'
 'model.layers.20.self_attn.k_proj' 'model.layers.20.self_attn.v_proj'
 'model